# Fraud Detection: Exploratory Data Analysis

## Business Problem
Credit card fraud costs banks billions annually. 
This notebook explores 284,807 credit card transactions 
to understand where, when and how fraud happens
before building a detection model.

## Dataset
- Source: Kaggle — ULB Credit Card Fraud Detection
- 284,807 transactions | 492 fraud cases (0.17%)
- Features: V1–V28 (PCA transformed), Amount, Time, Class

## Step 1: Load & Inspect the Data
We start by loading the dataset and doing a basic inspection.
We check:
- How many rows and columns we have
- Whether there are any missing values
- What data types each column is

In [1]:
import pandas as pd

df = pd.read_csv('/Users/mac/creditcard.csv')
df.shape

(284807, 31)

In [2]:
df.head()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


In [3]:
# Checking for missing values
df.isnull().sum()

Time      0
V1        0
V2        0
V3        0
V4        0
V5        0
V6        0
V7        0
V8        0
V9        0
V10       0
V11       0
V12       0
V13       0
V14       0
V15       0
V16       0
V17       0
V18       0
V19       0
V20       0
V21       0
V22       0
V23       0
V24       0
V25       0
V26       0
V27       0
V28       0
Amount    0
Class     0
dtype: int64

In [4]:
# Checking  data types
df.dtypes

Time      float64
V1        float64
V2        float64
V3        float64
V4        float64
V5        float64
V6        float64
V7        float64
V8        float64
V9        float64
V10       float64
V11       float64
V12       float64
V13       float64
V14       float64
V15       float64
V16       float64
V17       float64
V18       float64
V19       float64
V20       float64
V21       float64
V22       float64
V23       float64
V24       float64
V25       float64
V26       float64
V27       float64
V28       float64
Amount    float64
Class       int64
dtype: object

## Step 2: Class Imbalance
The most important discovery in this dataset.
We check how many transactions are fraud vs legitimate.

This matters because a model that predicts "legitimate" 
for every transaction would be 99.83% accurate but catch 
zero fraud, making accuracy a useless metric here.
We use Precision, Recall and F1 instead.

In [5]:
df['Class'].value_counts()

Class
0    284315
1       492
Name: count, dtype: int64

## Step 3: Transaction Amount Analysis
We compare transaction amounts between fraud and legitimate 
transactions.

Key insight: We look at the median not the mean — the mean 
can be misleading when there are extreme outliers. The median 
tells us what a TYPICAL transaction looks like for each class.

In [6]:
fraud = df[df['Class'] == 1]
legit = df[df['Class'] == 0]

print('Fraud transactions:')
print(fraud['Amount'].describe())

print('\nLegitimate transactions:')
print(legit['Amount'].describe())

Fraud transactions:
count     492.000000
mean      122.211321
std       256.683288
min         0.000000
25%         1.000000
50%         9.250000
75%       105.890000
max      2125.870000
Name: Amount, dtype: float64

Legitimate transactions:
count    284315.000000
mean         88.291022
std         250.105092
min           0.000000
25%           5.650000
50%          22.000000
75%          77.050000
max       25691.160000
Name: Amount, dtype: float64


## Step 4: Time Pattern Analysis
We convert the raw Time column (in seconds) into hours of 
the day to find when fraud is most likely to occur.

We look at fraud RATE by hour, not just raw count. 
A hour with 53 fraud out of 100 total transactions is far 
more dangerous than 53 fraud out of 50,000. Rate gives us 
the true picture.

In [7]:
df['Hour'] = (df['Time'] / 3600).astype(int) % 24

fraud_by_hour = df[df['Class']==1].groupby('Hour').size()
print(fraud_by_hour)

Hour
0      6
1     10
2     57
3     17
4     23
5     11
6      9
7     23
8      9
9     16
10     8
11    53
12    17
13    17
14    23
15    26
16    22
17    29
18    33
19    19
20    18
21    16
22     9
23    21
dtype: int64


In [8]:
legit_by_hour = df[df['Class']==0].groupby('Hour').size()

fraud_rate_by_hour = (fraud_by_hour / (fraud_by_hour + legit_by_hour) * 100).round(3)
print(fraud_rate_by_hour)

Hour
0     0.078
1     0.237
2     1.713
3     0.487
4     1.041
5     0.368
6     0.219
7     0.318
8     0.088
9     0.101
10    0.048
11    0.314
12    0.110
13    0.111
14    0.139
15    0.158
16    0.134
17    0.179
18    0.194
19    0.121
20    0.107
21    0.090
22    0.058
23    0.192
dtype: float64


## Key Takeaways

1. **Class Imbalance** : Only 0.17% of transactions are fraud. 
   Accuracy is misleading, we use Precision, Recall and F1.

2. **Small Transaction Pattern** :  Median fraud amount is $9.25 
   vs $22.00 for legitimate. Fraudsters test stolen cards with 
   small purchases first, known as card testing.

3. **Peak Fraud Hour** : 2am has the highest fraud rate at 1.713% 
   vs an average of 0.275%. Overnight hours need extra monitoring.

## Next Step
we prepare the data for modeling 
by scaling features and handling the class imbalance with SMOTE.